# 02 — Diagnóstico de resolución de entidades

Objetivo: identificar situaciones que podrían fragmentar proveedores o compradores en matrices y grafos. Este cuaderno lee `fact_award_supplier` publicado y **no modifica identificadores, tablas ni artefactos**. Sus resultados son candidatos para revisión, no correspondencias confirmadas.

La comparación de nombres aplica Unicode NFKC, minúsculas y normalización de espacios. No elimina acentos ni sufijos legales. Aunque dos nombres normalizados coincidan, sus identificadores pueden pertenecer a personas o entidades diferentes. Un mismo identificador con varios nombres tampoco demuestra por sí solo un error.


In [ ]:
import gzip
import json
import re
import unicodedata
from itertools import islice
from pathlib import Path

import pandas as pd
import pyarrow.dataset as ds
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
FACT_DIR = ROOT / 'data' / 'processed' / 'fact_award_supplier'
if not (FACT_DIR / '_SUCCESS').is_file():
    raise FileNotFoundError(f'Falta la tabla publicada: {FACT_DIR}')

columns = ['supplier_id', 'supplier_name', 'entity_id', 'entity_name']
dataset = ds.dataset(str(FACT_DIR), format='parquet', partitioning='hive')
fact = dataset.to_table(columns=columns).to_pandas()
print(f'Filas de adjudicación–proveedor analizadas: {len(fact):,}')


## Método y alcance

Construimos pares distintos identificador–nombre para cada rol y agrupamos solo los pares que tienen ambos valores. El conteo por nombre detecta nombres normalizados con varios identificadores; el conteo por identificador detecta variantes de nombre. Se conservan nombres e identificadores originales en las tablas de revisión. Los identificadores vacíos y nombres ausentes se reportan por separado.

La lectura en pandas es apropiada para el volumen actual y solo para exploración. Si crece el volumen, primero agregaremos los pares con Spark. Una futura correspondencia oficial se construirá con Spark, pruebas y evidencia verificable.


In [ ]:
def normalize_name(value):
    if pd.isna(value):
        return None
    normalized = unicodedata.normalize('NFKC', str(value)).casefold().strip()
    return re.sub(r'\s+', ' ', normalized) or None


def analyze_role(frame, role):
    id_column, name_column = f'{role}_id', f'{role}_name'
    pairs = frame[[id_column, name_column]].drop_duplicates().copy()
    pairs['id_clean'] = pairs[id_column].astype('string').str.strip().replace('', pd.NA)
    pairs['normalized_name'] = pairs[name_column].map(normalize_name)
    valid = pairs.dropna(subset=['id_clean', 'normalized_name'])
    by_name = (
        valid.groupby('normalized_name')['id_clean'].nunique().rename('identificadores')
    )
    by_id = valid.groupby('id_clean')['normalized_name'].nunique().rename('nombres')
    return pairs, valid, by_name[by_name > 1], by_id[by_id > 1]

diagnostics = {role: analyze_role(fact, role) for role in ['supplier', 'entity']}


## 1. Resumen de candidatos

Los valores ausentes se cuentan sobre pares distintos identificador–nombre. Los grupos con varios identificadores o nombres son alertas, no una estimación de duplicados reales.


In [ ]:
summary = []
for role, (pairs, _valid, names_multiple_ids, ids_multiple_names) in diagnostics.items():
    summary.append({
        'rol': role,
        'identificadores distintos': pairs['id_clean'].nunique(),
        'pares ID–nombre': len(pairs),
        'pares sin ID': int(pairs['id_clean'].isna().sum()),
        'pares sin nombre': int(pairs['normalized_name'].isna().sum()),
        'nombres con varios ID': len(names_multiple_ids),
        'ID con varios nombres': len(ids_multiple_names),
    })
display(pd.DataFrame(summary))


## 2. Casos para inspección

Se muestran como máximo 20 grupos por rol, ordenados por cantidad y nombre para facilitar una revisión reproducible. El listado no autoriza fusiones: habría que contrastar cada caso con evidencia adicional.


In [ ]:
for role, (_, valid, names_multiple_ids, ids_multiple_names) in diagnostics.items():
    print(f'\n{role}: nombres con varios identificadores')
    names = (
        names_multiple_ids.reset_index()
        .sort_values(['identificadores', 'normalized_name'], ascending=[False, True])
        .head(20)
    )
    name_details = (
        valid.groupby('normalized_name')[f'{role}_id']
        .agg(lambda values: sorted(set(values)))
        .rename('id_originales')
        .reset_index()
    )
    display(names.merge(name_details, on='normalized_name', how='left'))
    print(f'{role}: identificadores con varios nombres')
    ids = (
        ids_multiple_names.reset_index()
        .sort_values(['nombres', 'id_clean'], ascending=[False, True])
        .head(20)
    )
    id_details = (
        valid.groupby('id_clean')[f'{role}_name']
        .agg(lambda values: sorted(set(values)))
        .rename('nombres_originales')
        .reset_index()
    )
    raw_ids = (
        valid.groupby('id_clean')[f'{role}_id']
        .agg(lambda values: sorted(set(values)))
        .rename('id_originales')
        .reset_index()
    )
    display(ids.merge(id_details, on='id_clean').merge(raw_ids, on='id_clean'))


## 3. Evidencia de identificadores procesada con Spark

El resumen se calcula sobre referencias distintas por año, proceso, release, rol e identificador. `matched_one_party` indica una coincidencia en ese contexto; `unmatched` indica que no se encontró la parte y `multiple_party_records` requiere revisar la ambigüedad. Una coincidencia no constituye por sí misma una fusión.


In [ ]:
evidence_dir = ROOT / 'data' / 'processed' / 'identity_evidence'
if (evidence_dir / '_SUCCESS').is_file():
    with (evidence_dir / '_metadata.json').open(encoding='utf-8') as source:
        evidence = json.load(source)
    print('Referencias por rol y resultado del cruce:')
    display(pd.DataFrame(evidence['references_by_role_and_status']))
    print('Referencias con al menos un esquema e identificador completos:')
    display(pd.DataFrame(evidence['references_with_complete_identifier_by_role']))
    print('Referencias sin identificador completo vinculado:')
    display(pd.DataFrame(evidence.get('references_without_complete_identifier_by_role', [])))
    display(pd.DataFrame([
        {'métrica': 'identificadores de parties', 'valor': evidence['party_identifier_rows']},
        {'métrica': 'identificadores vinculados', 'valor': evidence['reference_identifier_rows']},
        {'métrica': 'grupos con identificador compartido',
         'valor': evidence['candidate_shared_identifier_groups']},
        {'métrica': 'grupos con variantes por esquema',
         'valor': evidence['candidate_identifier_variant_groups']},
    ]))
else:
    print('Sin evidencia publicada. Consulte la guía de réplica para generarla.')


## 4. Campos de identidad en una muestra de la fuente

Se examina el primer release con `parties` dentro de las primeras 101 líneas de cada archivo descargado. Esto descubre campos posibles; **no estima cobertura ni demuestra correspondencias**. La sección anterior resume la extracción completa con Spark.


In [ ]:
schema_samples = []
for path in sorted((ROOT / 'data' / 'raw').glob('*.jsonl.gz')):
    with gzip.open(path, 'rt', encoding='utf-8') as source:
        for line in islice(source, 101):
            release = json.loads(line)
            parties = release.get('parties') or []
            if not parties:
                continue
            party_fields = sorted({key for party in parties for key in party})
            identifiers = [p.get('identifier') for p in parties]
            identifier_fields = sorted({
                key for value in identifiers if isinstance(value, dict) for key in value
            })
            additional = [
                value for p in parties for value in (p.get('additionalIdentifiers') or [])
            ]
            additional_fields = sorted({
                key for value in additional if isinstance(value, dict) for key in value
            })
            schema_samples.append({
                'archivo': path.name,
                'campos parties': party_fields,
                'campos identifier': identifier_fields,
                'campos additionalIdentifiers': additional_fields,
            })
            break
if schema_samples:
    display(pd.DataFrame(schema_samples))
else:
    print('Sin muestra de parties: compruebe los archivos en data/raw/.')


## 5. Revisar referencias sin coincidencia en los releases originales

La tabla de Spark identifica referencias de proveedor sin `party.id` exacto. Aquí verificamos su contexto en los archivos originales, comprobando antes el SHA-256 contra los metadatos de la extracción. La lectura completa puede tardar varios minutos. No busca coincidencias aproximadas ni convierte un nombre igual en identidad confirmada. Si hay releases repetidos en el mismo contexto, se muestran todas las observaciones.


In [ ]:
import hashlib
from collections import defaultdict

if (evidence_dir / '_SUCCESS').is_file():
    matches = ds.dataset(str(evidence_dir / 'reference_matches'), format='parquet')
    unmatched = matches.to_table(filter=(ds.field('match_status') == 'unmatched')
                                 & (ds.field('reference_role') == 'supplier')).to_pandas()
    contexts = defaultdict(list)
    for row in unmatched.itertuples():
        contexts[(int(row.source_year), row.process_ocid, row.release_id)].append(row.reference_id)
    observations = []
    for source_info in evidence['sources']:
        year = int(source_info['year'])
        if not any(key[0] == year for key in contexts):
            continue
        path = ROOT / 'data' / 'raw' / source_info['filename']
        digest = hashlib.sha256()
        with path.open('rb') as source:
            for block in iter(lambda: source.read(1024 * 1024), b''):
                digest.update(block)
        if digest.hexdigest() != source_info['sha256']:
            raise ValueError(f'El archivo {path.name} no coincide con la extracción de evidencia.')
        with gzip.open(path, 'rt', encoding='utf-8') as source:
            for line in source:
                release = json.loads(line)
                key = (year, release.get('ocid'), release.get('id'))
                if key not in contexts:
                    continue
                parties = release.get('parties') or []
                suppliers = [s for a in (release.get('awards') or [])
                             for s in (a.get('suppliers') or [])]
                for reference_id in contexts[key]:
                    named = [s.get('name') for s in suppliers
                             if s.get('id') == reference_id and s.get('name')]
                    observations.append({
                        'source_year': year, 'process_ocid': key[1], 'release_id': key[2],
                        'reference_id': reference_id, 'num_parties': len(parties),
                        'supplier_en_awards': any(s.get('id') == reference_id for s in suppliers),
                        'party_id_exacto': any(p.get('id') == reference_id for p in parties),
                        'supplier_tiene_nombre': bool(named),
                        'party_nombre_exacto': any(p.get('name') in named for p in parties),
                    })
    reviewed = pd.DataFrame(observations)
    print(f'Referencias sin cruce: {len(unmatched)}; observaciones en fuente: {len(reviewed)}')
    if not reviewed.empty:
        display(reviewed.groupby('source_year').agg(
            observaciones=('reference_id', 'size'),
            supplier_en_awards=('supplier_en_awards', 'sum'),
            party_id_exacto=('party_id_exacto', 'sum'),
            con_nombre=('supplier_tiene_nombre', 'sum'),
            party_nombre_exacto=('party_nombre_exacto', 'sum'),
        ))
        display(reviewed.sort_values(['source_year', 'process_ocid', 'reference_id']))
    review_keys = ['source_year', 'process_ocid', 'release_id', 'reference_id']
    found_keys = (reviewed[review_keys].drop_duplicates() if not reviewed.empty
                  else unmatched[review_keys].iloc[0:0])
    not_found = unmatched.merge(found_keys, on=review_keys, how='left', indicator=True)
    not_found = not_found[not_found['_merge'] == 'left_only']
    print(f'Referencias sin release localizado: {len(not_found)}')
    if not not_found.empty:
        display(not_found)
else:
    print('Genere identity_evidence antes de ejecutar esta revisión.')


## 6. Muestra reproducible de nombres de proveedor compartidos

Revisamos los diez grupos con más ID, ordenando los empates por nombre normalizado. Se muestran los pares originales y los identificadores OCDS asociados por cruce exacto; la evidencia se presenta por ID, no como prueba de que los ID del grupo sean equivalentes. La muestra es intencional, no representativa. Un esquema `PE-RUC` no confirma por sí solo que el valor sea un RUC válido.


In [ ]:
if (evidence_dir / '_SUCCESS').is_file():
    _, valid_suppliers, supplier_names, _ = diagnostics['supplier']
    sample_names = (supplier_names.reset_index()
                    .sort_values(['identificadores', 'normalized_name'], ascending=[False, True])
                    .head(10))
    selected = valid_suppliers['normalized_name'].isin(sample_names['normalized_name'])
    sample_pairs = (valid_suppliers[selected]
                    [['normalized_name', 'supplier_id', 'supplier_name']]
                    .drop_duplicates()
                    .sort_values(['normalized_name', 'supplier_id', 'supplier_name']))
    sample_ids = sample_pairs['supplier_id'].dropna().unique().tolist()
    identifiers = ds.dataset(str(evidence_dir / 'reference_identifiers'), format='parquet')
    sample_evidence = identifiers.to_table(
        filter=(ds.field('reference_role') == 'supplier')
               & ds.field('reference_id').isin(sample_ids),
        columns=['reference_id', 'identifier_scheme', 'identifier_id'],
    ).to_pandas().drop_duplicates()
    sample_review = sample_pairs.merge(
        sample_evidence, left_on='supplier_id', right_on='reference_id', how='left')
    display(sample_names)
    print(f'ID en la muestra: {len(sample_ids)}; '
          f'ID con evidencia: {sample_evidence.reference_id.nunique()}')
    display(sample_review)
else:
    print('Genere identity_evidence antes de ejecutar esta revisión.')


## 7. Conclusiones y siguiente decisión

- La igualdad de nombres es una señal de revisión, no evidencia suficiente para fusionar identificadores, especialmente cuando ambos identificadores oficiales son diferentes.
- Las variantes de nombre bajo un mismo identificador se deben revisar sin cambiar la identidad ni borrar el nombre original.
- Antes de publicar una correspondencia canónica, definir fuentes de evidencia, reglas de aceptación y rechazo, ejemplos revisados y una tabla trazable `id_original → id_canonico` con motivo y estado.
- Las referencias sin cruce no justifican asignar otra identidad. Revisar su fuente y conservarlas como cobertura incompleta.
- Las tablas actuales y los futuros análisis deben indicar si usan identificadores originales o una correspondencia revisada. La resolución de entidades sigue pendiente hasta que exista esa correspondencia y se mida su efecto.
